# Segment Contribution Analysis

1. Aggregate transaction-level data.
To make the analysis more meaningful, I introduced controlled perturbations to simulate noticeable growth patterns.

2. Select a target merchant and benchmark key business metrics (e.g., transaction amount, transaction count) against peer merchants within the same category.

In [1]:
import pandas as pd
import numpy as np
from helper import get_generation

In [2]:
# Load dataset from huggingface
splits = {'train': 'credit_card_transaction_train.csv', 'test': 'credit_card_transaction_test.csv'}
df_all = pd.read_csv("hf://datasets/pointe77/credit-card-transaction/" + splits["train"], index_col=0)

/Users/yanxinye/github/competency-intelligence/ci/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ETL
df_all = df_all.assign(
    trans_date_trans_time=lambda x: pd.to_datetime(x['trans_date_trans_time']),
    trans_date=lambda x: x['trans_date_trans_time'].dt.strftime('%Y-%m-%d')
)

In [4]:
TARGET_MERCHANTS = "fraud_Kilback LLC"

# Date range of the analysis
START_DATE_TY = "2020-01-01"
START_DATE_LY = "2019-01-01"
END_DATE_TY = "2020-05-31"
END_DATE_LY = "2019-05-31"

DIMENSION_COLS = ['generation', 'state', 'gender', 'category']

In [5]:
df = df_all[
    (
        (df_all['trans_date'] >= START_DATE_TY) &
        (df_all['trans_date'] <= END_DATE_TY)
    )
    |
    (
        (df_all['trans_date'] >= START_DATE_LY) &
        (df_all['trans_date'] <= END_DATE_LY)
    )
].copy()

In [6]:
# Convert dob to datetime and extract generation
df = df.assign(
    dob=lambda x: pd.to_datetime(x['dob']),
    generation=lambda x: x['dob'].dt.year.map(get_generation)
)

In [7]:
df_comparison = (
    df.assign(
        yr_flag=lambda x: x['trans_date'].apply(
            lambda d: 'TY' if START_DATE_TY <= d <= END_DATE_TY
            else ('LY' if START_DATE_LY <= d <= END_DATE_LY else None)
        )
    )
    .groupby(DIMENSION_COLS + ['yr_flag', 'merchant'], as_index=False)['amt'].sum()
    .pivot_table(
        index= DIMENSION_COLS + ['merchant'],
        columns='yr_flag',
        values='amt',
        fill_value=0
    )
    .reset_index()
    .rename(columns={'TY': 'amt_ty', 'LY': 'amt_ly'})
    .rename_axis(None, axis=1)
)


In [8]:
# This data does not have any growth. Add some random variation to make it more interesting.

# Add more variation to amt_ty and amt_ly for bigger growth

np.random.seed(177)
random_factors_ty = np.clip(np.random.normal(0.3, 25, len(df_comparison)), -1, 3)
random_factors_ly = np.clip(np.random.normal(0, 2, len(df_comparison)), -0.5, 0.5)

df_comparison['amt_ty'] *= (1 + random_factors_ty)

df_comparison['amt_ly'] *= (1 + random_factors_ly)

In [9]:
def calculate_growth(df):
    assert 'amt_ty' in df.columns and 'amt_ly' in df.columns, "DataFrame must contain 'amt_ty' and 'amt_ly' columns"
    df['amt_diff'] = df['amt_ty'] - df['amt_ly']
    total_amt_diff = df['amt_diff'].sum()

    total_amt_ly = df['amt_ly'].sum()
    print(f"Total amount in LY: {total_amt_ly}")
    print(f"Total amount difference: {total_amt_diff}")
    print(f"Total growth: {total_amt_diff / total_amt_ly:.2%}")
    df["amt_growth_ctc"] = 1/total_amt_ly * df["amt_diff"]

    return df

#df_comparison = calculate_growth(df_comparison)

In [10]:
# df_comparison.head()

## Target vs Peers

In [11]:
target_industry = df_comparison[df_comparison['merchant'] == TARGET_MERCHANTS]['category'].drop_duplicates().to_list()
print(f"Target industry: {target_industry}")




df_target = df_comparison[df_comparison['merchant'] == TARGET_MERCHANTS].copy()
df_target = calculate_growth(df_target)

df_target.head()

Target industry: ['food_dining', 'grocery_pos']
Total amount in LY: 90116.45826107437
Total amount difference: 108492.6530555585
Total growth: 120.39%


,generation,state,gender,category,merchant,amt_ly,amt_ty,amt_diff,amt_growth_ctc
70,Boomer,AL,F,food_dining,fraud_Kilback LLC,323.040000,776.92,453.880000,0.005037
190,Boomer,AL,F,grocery_pos,fraud_Kilback LLC,213.750000,0.00,-213.750000,-0.002372
703,Boomer,AL,M,food_dining,fraud_Kilback LLC,191.370000,0.00,-191.370000,-0.002124
822,Boomer,AL,M,grocery_pos,fraud_Kilback LLC,68.841556,974.84,905.998444,0.010054
1314,Boomer,AR,F,food_dining,fraud_Kilback LLC,8.715000,0.00,-8.715000,-0.000097


In [12]:
df_peer = df_comparison[(df_comparison['merchant'] != TARGET_MERCHANTS) & (df_comparison['category'].isin(target_industry))].copy()
df_peer = calculate_growth(df_peer)
df_peer.head()

Total amount in LY: 4595809.920137372
Total amount difference: 4401093.053920547
Total growth: 95.76%


,generation,state,gender,category,merchant,amt_ly,amt_ty,amt_diff,amt_growth_ctc
49,Boomer,AL,F,food_dining,fraud_Abernathy and Sons,110.140000,0.000000,-110.140000,-0.000024
50,Boomer,AL,F,food_dining,"fraud_Armstrong, Walter and Gottlieb",215.340000,91.546459,-123.793541,-0.000027
51,Boomer,AL,F,food_dining,"fraud_Bahringer, Osinski and Block",202.155000,1417.120000,1214.965000,0.000264
52,Boomer,AL,F,food_dining,fraud_Bahringer-Streich,0.000000,559.160000,559.160000,0.000122
53,Boomer,AL,F,food_dining,fraud_Bechtelar-Rippin,219.466699,580.920000,361.453301,0.000079


In [17]:
peer_agg_metrics = {
    "amt_ty": ["mean"],
    "amt_ly": ["mean"],
}

peer_agg = df_peer.groupby(DIMENSION_COLS)[['amt_ty', 'amt_ly']].agg(peer_agg_metrics).reset_index()

peer_agg.columns = [
    f"{col[0]}_{col[1].lower()}".replace("_mean", "_peer") if col[1] else col[0].lower()
    for col in peer_agg.columns
]


total_amt_ly_peer = peer_agg["amt_ly_peer"].sum()
peer_agg = peer_agg.assign(
    amt_diff_peer=lambda x: x["amt_ty_peer"] - x["amt_ly_peer"],
    amt_growth_ctc_peer=lambda x: x["amt_diff_peer"] / total_amt_ly_peer,
    merchant=f"ex-{TARGET_MERCHANTS}"
)

peer_agg.head()
df_combined = pd.merge(df_target, peer_agg, on=DIMENSION_COLS, how='outer', suffixes=('_target', '_peer'))
df_combined['amt_growth_ctc_diff'] = (
    df_combined['amt_growth_ctc']
    .sub(df_combined['amt_growth_ctc_peer'], fill_value=0)
)

print(f"Average growth of target: {df_combined['amt_growth_ctc'].sum():.2%}")
print(f"Average growth of peers: {df_combined['amt_growth_ctc_peer'].sum():.2%}")
print(f"Average growth difference between target and peers: {df_combined['amt_growth_ctc_diff'].sum():.2%}")


Average growth of target: 120.39%
Average growth of peers: 94.52%
Average growth difference between target and peers: 25.87%


In [18]:
df_combined.sort_values(by='amt_growth_ctc_diff', ascending=False).head()

,generation,state,gender,category,merchant_target,amt_ly,amt_ty,amt_diff,amt_growth_ctc,amt_ty_peer,amt_ly_peer,amt_diff_peer,amt_growth_ctc_peer,merchant_peer,amt_growth_ctc_diff
545,Millennial,TX,F,grocery_pos,fraud_Kilback LLC,1192.993940,8094.64,6901.646060,0.076586,3895.925881,1786.410407,2109.515474,0.020291,ex-fraud_Kilback LLC,0.056295
269,Gen X,NY,F,grocery_pos,fraud_Kilback LLC,173.480000,6040.20,5866.720000,0.065102,2157.218700,937.360481,1219.858219,0.011734,ex-fraud_Kilback LLC,0.053368
167,Gen X,CA,F,grocery_pos,fraud_Kilback LLC,888.458749,5724.92,4836.461251,0.053669,1583.183763,964.689093,618.494670,0.005949,ex-fraud_Kilback LLC,0.047720
531,Millennial,PA,F,grocery_pos,fraud_Kilback LLC,851.775000,4752.72,3900.945000,0.043288,1966.006811,934.293435,1031.713376,0.009924,ex-fraud_Kilback LLC,0.033364
439,Millennial,GA,M,grocery_pos,fraud_Kilback LLC,149.175000,3326.92,3177.745000,0.035263,387.123329,134.481102,252.642227,0.002430,ex-fraud_Kilback LLC,0.032833
